# Memory Bank in STNMamba

The memory bank stores feature patterns that can be retrieved to reconstruct input features and enhance their spatial-temporal representations. It consists of three main operations: **reading, writing, and memory enhancement**.

## 1. Feature Representation

Given a spatial-temporal feature map:

\begin{equation}
F_{st}^{i}\in\mathbb{R}^{H\times W\times C}
\end{equation}

The feature map is reshaped into a set of query vectors:

\begin{equation}
Q=\{q_1,\ldots,q_{\hat N}\}
\in\mathbb{R}^{\hat N\times C},
\qquad \hat N=H\times W
\end{equation}

Each query $q_i$ represents the features at a spatial position. The memory bank contains $N$ memory items, denoted by $M=\{p_1,\ldots,p_N\}$, which store learned feature patterns.

## 2. Reading Operation

The reading operation retrieves relevant memory items to reconstruct each query.

First, cosine similarity is calculated between each query $q_i$ and memory item $p_j$:

\begin{equation}
\hat{w}_i^{(j)}
=
\frac{q_i p_j^T}
{\|q_i\|_2\|p_j\|_2}
\end{equation}

A **Top-k scheme** retains only the top $k\%$ largest similarity values, setting the remaining values to zero. This restricts the reconstruction to the most relevant memory items.

Then, softmax converts the retained similarities into normalized weights:

\begin{equation}
w_i^{(j)}
=
\frac{\exp(\hat{w}_i^{(j)})}
{\displaystyle\sum_{\ell=1}^{N}
\exp(\hat{w}_i^{(\ell)})}
\end{equation}

Finally, the reconstructed query is obtained through a weighted combination of memory items:

\begin{equation}
\hat{q}_i=w_iM
\end{equation}

Thus, each query is reconstructed using the memory patterns most similar to its features.

## 3. Writing Operation

The writing operation updates the memory bank using the current query features $Q$.

For each memory item $m_j$, cosine similarities are calculated against all queries. A softmax operation produces the weights $\tilde{w}_j$, which determine the contribution of the queries to the memory update.

The updated memory item is computed as:

\begin{equation}
\tilde{m}_j
=
\operatorname{L2}
\left(m_j+\tilde{w}_jQ\right)
\end{equation}

Here, $m_j$ represents the previous memory item, while $\tilde{w}_jQ$ aggregates information from the current queries. L2 normalization controls the scale of the updated memory vector.

This operation allows the memory to retain historical information while incorporating new feature information.

## 4. Memory Enhancement

After reading, the reconstructed queries are reshaped into the memory feature map $F_m^i$. This feature map is combined with the original spatial-temporal representation:

\begin{equation}
\tilde{F}_{st}^{i}
=
F_m^{i}+s\odot F_{st}^{i}
\end{equation}

where:
- $F_m^i$ is the retrieved memory feature map.
- $F_{st}^i$ is the original input feature map.
- $s\in\mathbb{R}^{C}$ is a trainable coefficient that balances the contribution of the original features.
- $\odot$ denotes element-wise multiplication, with $s$ applied across channels.

Unlike approaches that concatenate features or use only reconstructed features, this operation combines memory information with the original representation while allowing the model to learn the contribution of each channel.

## 5. Relevance to Video Anomaly Detection

The memory bank can support anomaly detection by providing a reference of learned feature patterns.

- **Normal patterns:** If a query resembles stored patterns, the memory can reconstruct its features relatively well.
- **Anomalous patterns:** If a query differs significantly from stored patterns, its reconstruction may be less accurate, potentially providing a useful anomaly signal.

However, this behavior is not guaranteed. Anomalies may share features with normal patterns, and the memory bank may partially reconstruct them. The memory enhancement operation itself is not an anomaly score; its contribution depends on the complete architecture and detection mechanism.

## Summary

The memory bank retrieves relevant patterns to reconstruct input features, combines the reconstructed information with the original spatial-temporal features, and updates its stored representations using weighted query information. Its potential benefit for anomaly detection comes from how memory-based reconstruction influences feature representations, rather than from an assumption that every anomaly must be reconstructed poorly.